# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Data Entry

In [ ]:
df = pd.read_csv("Iris.csv")

# Select features
sepal_features = ["sepal.length", "sepal.width"]
petal_features = ["petal.length", "petal.width"]

# Target column
target = "variety"

# Split Dataset into Training and Testing

In [ ]:
X = df[sepal_features + petal_features]
y = df[target]

# 80% Training and 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split Training into Training and Validation
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)


# F1 Score Calculation

In [ ]:
def calculate_f1(y_true, y_pred):

    classes = np.unique(y_true)

    f1_scores = []

    for cls in classes:

        # Calculate True Positive
        TP = np.sum((y_true == cls) & (y_pred == cls))

        # Calculate False Positive
        FP = np.sum((y_true != cls) & (y_pred == cls))

        # Calculate False Negative
        FN = np.sum((y_true == cls) & (y_pred != cls))

        # Calculate Precision
        if TP + FP == 0:
            precision = 0
        else:
            precision = TP / (TP + FP)

        # Calculate Recall
        if TP + FN == 0:
            recall = 0
        else:
            recall = TP / (TP + FN)

        # Calculate F1 Score
        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * (precision * recall) / (precision + recall)

        f1_scores.append(f1)

    # Calculate Macro F1
    return np.mean(f1_scores)



# Define K Values

In [ ]:
k_values = range(1, 31)

# Find Best K for Sepal Features

In [ ]:
sepal_f1_scores = []

for k in k_values:

    # Create KNN Model
    sepal_model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    # Train Model
    sepal_model.fit(
        X_fit[sepal_features],
        y_fit
    )

    # Predict Validation Data
    predictions = sepal_model.predict(
        X_val[sepal_features]
    )

    # Calculate F1 Score
    f1 = calculate_f1(
        y_val.to_numpy(),
        predictions
    )

    sepal_f1_scores.append(f1)


# Find Best K
best_sepal_index = np.argmax(sepal_f1_scores)

best_sepal_k = list(k_values)[best_sepal_index]

print("Best K for Sepal:", best_sepal_k)
print("Best Sepal Validation F1:", sepal_f1_scores[best_sepal_index])



# Find Best K for Petal Features

In [ ]:
petal_f1_scores = []

for k in k_values:

    # Create KNN Model
    petal_model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    # Train Model
    petal_model.fit(
        X_fit[petal_features],
        y_fit
    )

    # Predict Validation Data
    predictions = petal_model.predict(
        X_val[petal_features]
    )

    # Calculate F1 Score
    f1 = calculate_f1(
        y_val.to_numpy(),
        predictions
    )

    petal_f1_scores.append(f1)


# Find Best K
best_petal_index = np.argmax(petal_f1_scores)

best_petal_k = list(k_values)[best_petal_index]

print("\nBest K for Petal:", best_petal_k)
print("Best Petal Validation F1:", petal_f1_scores[best_petal_index])


# Train Final Models Using Best K

In [ ]:
# Final Sepal Model
sepal_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_sepal_k))
])

# Final Petal Model
petal_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_petal_k))
])

# Train on the full training dataset
sepal_model.fit(
    X_train[sepal_features],
    y_train
)

petal_model.fit(
    X_train[petal_features],
    y_train
)



# Test Final Models

In [ ]:
# Sepal Predictions
sepal_predictions = sepal_model.predict(
    X_test[sepal_features]
)

# Petal Predictions
petal_predictions = petal_model.predict(
    X_test[petal_features]
)

# Calculate Test F1
sepal_test_f1 = calculate_f1(
    y_test.to_numpy(),
    sepal_predictions
)

petal_test_f1 = calculate_f1(
    y_test.to_numpy(),
    petal_predictions
)

print("\n========== Final Test Results ==========")

print("Sepal Best K:", best_sepal_k)
print("Sepal Test F1:", sepal_test_f1)

print("\nPetal Best K:", best_petal_k)
print("Petal Test F1:", petal_test_f1)

# Plot K vs F1 Score

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    list(k_values),
    sepal_f1_scores,
    marker="o",
    color="#2563EB",
    label="Sepal"
)

plt.plot(
    list(k_values),
    petal_f1_scores,
    marker="s",
    color="#F97316",
    label="Petal"
)

# Highlight Best K for Sepal
plt.scatter(
    best_sepal_k,
    sepal_f1_scores[best_sepal_index],
    color="#2563EB",
    marker="*",
    s=200,
    label="Best Sepal K",
    zorder=3
)

# Highlight Best K for Petal
plt.scatter(
    best_petal_k,
    petal_f1_scores[best_petal_index],
    color="#F97316",
    marker="*",
    s=200,
    label="Best Petal K",
    zorder=3
)

plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Macro F1 Score")

plt.title("Finding Best K Using Manual F1 Calculation")

plt.xticks(list(k_values))
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# Plot Decision Boundaries and Data Points

In [ ]:
def plot_knn(model, features, title, ax):

    # Set plot boundaries
    x_min = df[features[0]].min() - 0.5
    x_max = df[features[0]].max() + 0.5

    y_min = df[features[1]].min() - 0.5
    y_max = df[features[1]].max() + 0.5

    # Create a grid
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.02),
        np.arange(y_min, y_max, 0.02)
    )

    # Prepare grid data
    grid = pd.DataFrame({
        features[0]: xx.ravel(),
        features[1]: yy.ravel()
    })

    # Predict classes for grid points
    Z = model.predict(grid)

    # Convert class names to numbers
    classes = model.classes_

    class_map = {
        label: i for i, label in enumerate(classes)
    }

    Z = pd.Series(Z).map(class_map).to_numpy()

    Z = Z.reshape(xx.shape)

    # Define colors
    colors = ["#2563EB", "#F97316", "#16A34A"]

    # Plot decision boundaries
    ax.contourf(
        xx,
        yy,
        Z,
        levels=np.arange(len(classes) + 1) - 0.5,
        colors=colors[:len(classes)],
        alpha=0.15
    )

    # Plot Training and Testing Data
    for i, variety in enumerate(classes):

        # Training Data
        train_subset = X_train[
            y_train == variety
        ]

        ax.scatter(
            train_subset[features[0]],
            train_subset[features[1]],
            color=colors[i],
            marker="o",
            edgecolors="black",
            s=55,
            alpha=0.85,
            label=f"{variety} - Train"
        )

        # Testing Data
        test_subset = X_test[
            y_test == variety
        ]

        ax.scatter(
            test_subset[features[0]],
            test_subset[features[1]],
            color=colors[i],
            marker="X",
            edgecolors="black",
            linewidths=0.8,
            s=110,
            alpha=1,
            label=f"{variety} - Test"
        )

    # Plot settings
    ax.set_xlabel(features[0], fontsize=11)
    ax.set_ylabel(features[1], fontsize=11)

    ax.set_title(title, fontsize=13, fontweight="bold")

    ax.legend(
        title="Dataset",
        fontsize=8
    )

    ax.grid(
        True,
        linestyle="--",
        alpha=0.25
    )


# Create Final Plots

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(17, 7)
)

# Sepal Plot
plot_knn(
    sepal_model,
    sepal_features,
    f"Sepal KNN (Best K = {best_sepal_k})",
    axes[0]
)

# Petal Plot
plot_knn(
    petal_model,
    petal_features,
    f"Petal KNN (Best K = {best_petal_k})",
    axes[1]
)

plt.tight_layout()
plt.show()